# Build a molecule library

Turns a CSV of SMILES into the LMDB the screening notebooks read, using every
available core.

Run this **once** per compound set. All six screening notebooks then reuse the
result instead of rebuilding it, which matters because conformer generation is
the slowest step and does not depend on the pocket, the weights, or the method.

### Why it is worth separating

Conformer generation runs at roughly **0.16 s per molecule** single-threaded -
about 20 hours for 460,000 molecules. It is pure RDKit and embarrassingly
parallel, so it scales almost linearly with cores.

| | single-threaded | parallel |
| --- | --- | --- |
| 12 cores | 20.5 h | ~2 h |
| 24 vCPU VM | 20.5 h | ~50 min |

A GPU does not help here - this is CPU work whatever the hardware. With encoding
moved to a GPU, this becomes the bottleneck.

### Naming

The LMDB is named after the CSV, so different compound sets never collide:

```
pgk2_ligands.csv  ->  pgk2_ligands.lmdb  ->  emb/pgk2_ligands.lmdb.pkl
```

That last part matters: the embedding cache is keyed on the LMDB's filename, so
naming by source makes the cache correct per library rather than something to
remember to clear.

## Configuration

In [ ]:
from pathlib import Path
import os

# Found by walking up from wherever this runs, so the same notebook works on
# any machine and survives a git pull. unimol/ marks the repo root.
def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "unimol").is_dir():
            return p
    raise RuntimeError(
        f"no parent of {Path.cwd()} contains unimol/, so the repo root could "
        "not be found. Some kernels start outside the repo - set REPO to the "
        "clone path on the next line instead.")

REPO     = _find_repo()
# REPO   = Path("/path/to/DrugCLIP")      # uncomment if the above cannot find it
PIPELINE = REPO.parent / "DrugCLIP_screen_pipeline"
MOL_CSV = REPO / "data" / "pgk2" / "pgk2_ligands.csv"
OUT_DIR = REPO / "data" / "lmdb"          # shared by every screening notebook

N_CONF    = 10                             # conformers per molecule
N_WORKERS = max(1, os.cpu_count() - 4)     # headroom for the kernel, the editor
                                           # and the OS; with only two spare the
                                           # machine stops responding and remote
                                           # terminals drop
PROGRESS  = Path("build_progress.log")     # live progress. nbconvert buffers a
                                           # cell's output until the cell ends, so
                                           # a detached run shows nothing without
                                           # somewhere separate to write
CHUNK     = None                           # None = sized from the library (see below)

SMILES_COL = None    # None = autodetect ("smiles"/"SMILES"), or name it
NAME_COL   = None    # None = autodetect, falls back to row index

OUT_DIR.mkdir(parents=True, exist_ok=True)
MOLS_LMDB = OUT_DIR / (MOL_CSV.stem + ".lmdb")

print("source :", MOL_CSV)
print("output :", MOLS_LMDB)
print("workers:", N_WORKERS, "of", os.cpu_count(), "cores")

## Read the CSV

Column names vary between compound sets, so they are detected rather than
assumed. A file with only SMILES is fine - names fall back to the row index.

In [ ]:
import pandas as pd

df = pd.read_csv(MOL_CSV)
print("columns:", list(df.columns), "| rows:", len(df))

smi_col = SMILES_COL or next(
    (c for c in df.columns if c.strip().lower() in ("smiles", "smi", "canonical_smiles")),
    None)
assert smi_col, f"no SMILES column found in {list(df.columns)} - set SMILES_COL"

name_col = NAME_COL or next(
    (c for c in df.columns if c.strip().lower() in ("name", "id", "molname", "hitid")),
    None)

pairs = [(str(r[name_col]) if name_col else f"mol_{i}", str(r[smi_col]))
         for i, r in df.iterrows()]
print(f"using smiles='{smi_col}', name={name_col or '(row index)'}")
print("first:", pairs[0])

## Generate conformers in parallel

The recipe matches the upstream scripts: add hydrogens, embed several
conformers, relax with MMFF, then drop the hydrogens again.

Hydrogens are scaffolding - they are needed for the geometry to come out right,
and removing them afterwards leaves the heavy atoms where the hydrogen-aware
optimisation put them. The model strips hydrogens anyway.

`pruneRmsThresh=1` discards conformers within 1 A of one another, so the ten
kept are genuinely different shapes rather than near-duplicates.

In [ ]:
import numpy as np
from multiprocessing import Pool
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem


def _one(arg):
    name, smi, n_conf = arg
    RDLogger.DisableLog("rdApp.*")          # each worker silences separately
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None, (name, "unparseable SMILES")
    mol = Chem.AddHs(mol)
    if len(AllChem.EmbedMultipleConfs(mol, numConfs=n_conf, randomSeed=42,
                                      pruneRmsThresh=1)) == 0:
        return None, (name, "no conformer embedded")
    try:
        AllChem.MMFFOptimizeMoleculeConfs(mol)
    except Exception:
        pass                                 # keep the unoptimised geometry
    mol = Chem.RemoveHs(mol)
    return {"atoms": [a.GetSymbol() for a in mol.GetAtoms()],
            "coordinates": [np.array(mol.GetConformer(c).GetPositions(),
                                     dtype=np.float32)
                            for c in range(mol.GetNumConformers())],
            "smi": smi, "name": name}, None


def build(pairs, n_workers, n_conf, chunk=None):
    tasks = [(n, s, n_conf) for n, s in pairs]

    # Give every worker several tasks. A fixed chunk starves most of them on a
    # small library: 212 molecules at chunksize 200 is 2 tasks, so 2 workers
    # do the work and the rest idle.
    if chunk is None:
        chunk = max(1, min(200, len(tasks) // (n_workers * 4) or 1))
    print(f"{len(tasks)} molecules, chunksize {chunk} "
          f"-> {-(-len(tasks) // chunk)} tasks across {n_workers} workers")

    recs, fails = [], []
    with Pool(n_workers) as pool:
        stream = open(PROGRESS, "w", buffering=1) if PROGRESS else None
        for rec, err in tqdm(pool.imap_unordered(_one, tasks, chunksize=chunk),
                             total=len(tasks), desc="conformers",
                             file=stream, mininterval=5):
            (recs.append(rec) if rec else fails.append(err))
        if stream:
            stream.close()
    return recs, fails


from tqdm.auto import tqdm
import time

t0 = time.time()
records, failures = build(pairs, N_WORKERS, N_CONF, CHUNK)
dt = time.time() - t0

print(f"\nbuilt {len(records)} | failed {len(failures)} | {dt / 60:.1f} min")
print(f"{dt / max(len(pairs), 1):.3f} s per molecule across {N_WORKERS} workers")
for f in failures[:10]:
    print("  FAILED:", f)
if len(failures) > 10:
    print(f"  ... and {len(failures) - 10} more")

## Check the atom types

Any element missing from `dict_mol.txt` silently becomes `[UNK]`, so the model
sees an unknown atom rather than failing. Worth knowing about before screening
rather than after.

In [ ]:
vocab = set((REPO / "data" / "dict_mol.txt").read_text().split())
seen = set(a for r in records for a in r["atoms"])
unknown = seen - vocab
print("elements present:", sorted(seen))
print("not in dict_mol.txt (become [UNK]):", sorted(unknown) or "none")
if unknown:
    n = sum(any(a in unknown for a in r["atoms"]) for r in records)
    print(f"  affects {n} of {len(records)} molecules")

## Write the LMDB

In [ ]:
import pickle, lmdb

if MOLS_LMDB.exists():
    MOLS_LMDB.unlink()

env = lmdb.open(str(MOLS_LMDB), subdir=False, readonly=False, lock=False,
                readahead=False, meminit=False, map_size=1099511627776)
with env.begin(write=True) as txn:
    for i, r in enumerate(records):
        txn.put(str(i).encode("ascii"), pickle.dumps(r))
    # provenance, so a notebook can tell what a reused LMDB actually contains
    txn.put(b"__meta__", pickle.dumps({
        "source_csv": MOL_CSV.name,
        "n_molecules": len(records),
        "n_conformers": N_CONF,
    }))
env.close()

size = MOLS_LMDB.stat().st_size / 1e6
print(f"wrote {MOLS_LMDB}  ({size:.1f} MB, {len(records)} molecules)")

In [ ]:
# read it back
env = lmdb.open(str(MOLS_LMDB), subdir=False, readonly=True, lock=False,
                readahead=False, meminit=False, max_readers=256)
with env.begin() as txn:
    meta = pickle.loads(txn.get(b"__meta__"))
    first = pickle.loads(txn.get(b"0"))
env.close()

print("meta:", meta)
print(f"first record: {first['name']}  {len(first['atoms'])} atoms, "
      f"{len(first['coordinates'])} conformers, "
      f"shape {first['coordinates'][0].shape}")

## Next

Point any screening notebook at the same CSV. They look for
`data/lmdb/<csv-stem>.lmdb` and reuse it rather than regenerating, so the cost
above is paid once no matter how many methods or folds are run.